# Avance 1 — EDA (Clientes)

**Estudiante:** Lourdes Diamela Alarcon

_Actualizado: 2025-09-17_

## Descripción
Se cargan, limpian y exploran los datos de clientes. El análisis se centra en **Estados Unidos (USA)** y, para profundizar, en la ciudad de **Miami**. Se documentan decisiones de calidad de datos y se generan visualizaciones y tablas de resumen.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
%matplotlib inline

RAW_PATH = '../data/base_datos_restaurantes_USA_v2.csv'
df = pd.read_csv(RAW_PATH)
print('Shape crudo:', df.shape)
df.head(5)


### Calidad de datos (tipos, nulos, duplicados)

In [ ]:

display(df.dtypes)
nulls = (df.isna().mean()*100).round(2).sort_values(ascending=False)
print('\n% de nulos por columna:')
display(nulls.head(15))
print('Duplicados:', df.duplicated().sum())


### Filtrado por país y normalizaciones

In [ ]:

# El dataset corresponde a EE.UU.; se explicita para el análisis por país
df['pais'] = 'USA'
df = df[df['pais']=='USA'].copy()

# Casting numérico y normalización de texto
for c in ['edad','frecuencia_visita','promedio_gasto_comida']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

for c in ['ciudad_residencia','genero','preferencias_alimenticias','metodo_pago']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

# Eliminar duplicados exactos
df = df.drop_duplicates().reset_index(drop=True)
print('Shape post-limpieza base:', df.shape)


### Detección de valores atípicos evidentes

In [ ]:

# Ejemplo de outlier extremo en edad: valores > 100 se marcan para revisión
if 'edad' in df.columns:
    outlier_mask = df['edad'] > 100
    print('Posibles outliers de edad (>100):', int(outlier_mask.sum()))
    # Se excluyen del análisis descriptivo para no sesgar métricas
    df.loc[outlier_mask, 'edad'] = np.nan


### Filtro por ciudad para integrar con Yelp

In [ ]:

CIUDAD_OBJETIVO = 'Miami'
if 'ciudad_residencia' in df.columns:
    df_city = df[df['ciudad_residencia'].str.lower()==CIUDAD_OBJETIVO.lower()].copy()
else:
    df_city = df.copy()
print('Filas en ciudad objetivo:', df_city.shape[0])
df_city['ciudad_residencia'].value_counts().head()


### Distribuciones y tablas de resumen

In [ ]:

# Histogramas de variables numéricas
num_cols = [c for c in ['edad','frecuencia_visita','promedio_gasto_comida'] if c in df_city.columns]
for c in num_cols:
    plt.figure(figsize=(6,4))
    df_city[c].dropna().hist(bins=30)
    plt.title(f'Distribución de {c} — {CIUDAD_OBJETIVO}')
    plt.xlabel(c); plt.ylabel('Frecuencia')
    plt.tight_layout()
    plt.savefig(f'../reports/figures/dist_{c}.png', bbox_inches='tight')
    plt.show()

# Categóricas principales
if 'genero' in df_city.columns:
    plt.figure(figsize=(6,4))
    df_city['genero'].value_counts(dropna=False).plot(kind='bar')
    plt.title(f'Distribución por género — {CIUDAD_OBJETIVO}')
    plt.ylabel('Conteo')
    plt.tight_layout()
    plt.savefig('../reports/figures/dist_genero.png', bbox_inches='tight')
    plt.show()

# Resúmenes estadísticos
resumen = {}
if set(['frecuencia_visita','promedio_gasto_comida']).issubset(df_city.columns):
    resumen['stats'] = df_city[['frecuencia_visita','promedio_gasto_comida']].describe()
resumen


### Exportación de datasets procesados

In [ ]:

df.to_csv('../data/processed_clientes_clean.csv', index=False)
df_city.to_csv('../data/processed_clientes_city.csv', index=False)
print('Exportados: processed_clientes_clean.csv | processed_clientes_city.csv')
